In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix

warnings.filterwarnings('ignore')

print(f'pandas      {pd.__version__}')
print(f'matplotlib  {matplotlib.__version__}')
print(f'seaborn     {sns.__version__}')

In [ ]:
BASE_DIR    = Path(r'')
RESULTS_DIR = BASE_DIR / 'Results'
CKPT_DIR    = RESULTS_DIR / 'eval_checkpoints'
FIGURES_DIR = RESULTS_DIR / 'Figures' / 'Results_Analysis'
TABLES_DIR  = RESULTS_DIR / 'Tables'

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

METRICS_PATH = RESULTS_DIR / 'model_evaluation_results.json'

matplotlib.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('Setup complete. Visual styling initialized.')
print(f'Metrics file : {METRICS_PATH}')
print(f'Checkpoint dir: {CKPT_DIR}')
print(f'Figures dir   : {FIGURES_DIR}')
print(f'Tables dir    : {TABLES_DIR}')

## Section 1 — Load Evaluation Metrics


In [ ]:
try:
    with open(METRICS_PATH, 'r', encoding='utf-8') as f:
        eval_data = json.load(f)

    metrics = eval_data.get('models', {})
    df_metrics = pd.DataFrame(metrics).T

    numeric_cols = [
        'cls_acc', 'cls_f1_macro', 'adv_acc', 'adv_acc_answered_only',
        'inconclusive'
    ]
    for col in numeric_cols:
        if col in df_metrics.columns:
            df_metrics[col] = pd.to_numeric(df_metrics[col], errors='coerce')

    df_metrics = df_metrics.dropna(subset=['adv_acc'])
    df_metrics = df_metrics.sort_values('adv_acc', ascending=True)

    print('=== Final Model Metrics Loaded ===')
    cols_show = [
        c for c in [
            'type', 'cls_acc', 'cls_f1_macro', 'adv_acc',
            'adv_acc_answered_only', 'inconclusive',
            'cls_acc_ci95', 'cls_f1_ci95', 'adv_acc_ci95'
        ] if c in df_metrics.columns
    ]
    print(df_metrics[cols_show].to_string())

except FileNotFoundError:
    raise FileNotFoundError('Error: run NB-04 first to generate model_evaluation_results.json')

## Section 2 — Figure R01 (The Reasoning Gap Bar Chart)
Demonstrates the bump in SOTA performance from Zero-Shot Classification to Adversarial Selection.


In [ ]:
df_bar = df_metrics.dropna(subset=['cls_acc']).copy()

models = list(df_bar.index)
cls_acc = df_bar['cls_acc'].values * 100
adv_acc = df_bar['adv_acc'].values * 100

x = np.arange(len(models))
width = 0.40

fig, ax = plt.subplots(figsize=(12, 7))
bars1 = ax.bar(
    x - width/2, cls_acc, width,
    label='JACV-CLS',
    color='#2C3E50',
    alpha=0.9
)
bars2 = ax.bar(
    x + width/2, adv_acc, width,
    label='JACV-ADV',
    color='#27AE60',
    alpha=0.9
)

ax.set_ylabel('Accuracy (%)', fontweight='bold', fontsize=12)
ax.set_title('Figure R01 — The Reasoning Gap:\nClassification vs. Pair Selection', pad=20, fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(models, fontweight='bold', fontsize=10, rotation=45, ha='right')
ax.set_ylim(0, 115)
ax.legend(loc='upper left', frameon=False)
ax.grid(axis='y', linestyle='--', alpha=0.3)

def label_bars(bars):
    for bar in bars:
        height = bar.get_height()
        ax.annotate(
            f'{height:.1f}%',
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=9,
            fontweight='bold',
            rotation=90
        )

label_bars(bars1)
label_bars(bars2)

plt.tight_layout()
p = FIGURES_DIR / 'figR01_reasoning_gap.png'
fig.savefig(p, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved -> {p}')

## Section 3 — Load LLM Raw Predictions (For Confusion Matrix)


In [ ]:
cls_ckpts = {}
raw_ckpts = {}

if not CKPT_DIR.exists():
    print('Checkpoint directory not found.')
else:
    for fpath in CKPT_DIR.glob('ckpt_*.json'):
        model_name = fpath.stem.replace('ckpt_', '').replace('_', ' ')
        try:
            with open(fpath, 'r', encoding='utf-8') as f:
                data = json.load(f)

            raw_ckpts[model_name] = data

            if 'cls' in data and data['cls']:
                cls_ckpts[model_name] = data['cls']

        except Exception as e:
            print(f'Could not load {fpath.name}: {e}')

print(f'Loaded checkpoints for: {list(raw_ckpts.keys())}')

## Section 4 — Figure R02 (JACV-CLS Confusion Matrices)
Visually mapping the errors of each LLM individually and exporting them.


In [ ]:
labels_order = ['COHERENT', 'INCOHERENT', 'CONTRADICTORY']

if not cls_ckpts:
    print('No CLS predictions found in checkpoints folder.')
else:
    for model_name, cls_preds in cls_ckpts.items():
        y_true, y_pred = [], []

        for _, v in cls_preds.items():
            y_true.append(v['gold'])
            pred = str(v['pred']).strip().upper()
            if pred in labels_order:
                y_pred.append(pred)
            else:
                y_pred.append('ERROR')

        valid_idx = [i for i, p in enumerate(y_pred) if p in labels_order]
        yt_clean = [y_true[i] for i in valid_idx]
        yp_clean = [y_pred[i] for i in valid_idx]

        if not yt_clean:
            print(f'Skipping {model_name}: no valid CLS predictions.')
            continue

        cm = confusion_matrix(yt_clean, yp_clean, labels=labels_order)

        fig, ax = plt.subplots(figsize=(6, 5))
        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            ax=ax,
            cbar=False,
            xticklabels=labels_order,
            yticklabels=labels_order,
            annot_kws={'size': 12, 'weight': 'bold'}
        )

        ax.set_title(f'{model_name}\nJACV-CLS Confusion Matrix', weight='bold', pad=15)
        ax.set_xlabel('Predicted Label', weight='bold')
        ax.set_ylabel('True (Gold) Label', weight='bold')
        ax.tick_params(axis='x', rotation=45)

        plt.tight_layout()
        p = FIGURES_DIR / f'figR02_CM_{model_name.replace(" ", "-")}.png'
        fig.savefig(p, bbox_inches='tight', facecolor='white')
        plt.show()
        print(f'Saved -> {p}')

## Section 5 — Figure R03 (Cognitive Prompting Lift)\nMeasuring the impact of Legal-CoT vs. Zero-Shot on the Adversarial Task.


In [ ]:
cot_models = [m for m in df_metrics.index if '-CoT' in m]

if not cot_models:
    print('No CoT models found in metrics. Run Section 8 of NB-04 first.')
else:
    bases = [m.replace('-CoT', '') for m in cot_models]
    valid_pairs = []

    for b, c in zip(bases, cot_models):
        if b in df_metrics.index:
            valid_pairs.append((b, c))

    if valid_pairs:
        fig, ax = plt.subplots(figsize=(8, 6))
        x = np.arange(len(valid_pairs))
        width = 0.35

        acc_base = [df_metrics.loc[b, 'adv_acc'] * 100 for b, c in valid_pairs]
        acc_cot  = [df_metrics.loc[c, 'adv_acc'] * 100 for b, c in valid_pairs]

        bars1 = ax.bar(x - width/2, acc_base, width, label='Zero-Shot', color='#34495E')
        bars2 = ax.bar(x + width/2, acc_cot, width, label='Legal-CoT', color='#E74C3C')

        ax.set_ylabel('JACV-ADV Accuracy (%)', weight='bold')
        ax.set_title('Figure R03 — Impact of Cognitive Prompting (Legal-CoT)', pad=15, weight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels([b for b, c in valid_pairs], weight='bold')
        ax.set_ylim(0, 115)
        ax.legend(frameon=False)

        def annotate(bars):
            for bar in bars:
                h = bar.get_height()
                ax.annotate(
                    f'{h:.1f}%',
                    xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3),
                    textcoords='offset points',
                    ha='center',
                    weight='bold'
                )

        annotate(bars1)
        annotate(bars2)

        plt.tight_layout()
        p = FIGURES_DIR / 'figR03_cognitive_lift.png'
        fig.savefig(p, bbox_inches='tight', facecolor='white')
        plt.show()
        print(f'Saved -> {p}')

        if 'adv_acc_answered_only' in df_metrics.columns:
            subset = df_metrics.loc[[c for _, c in valid_pairs]].copy()
            subset = subset.dropna(subset=['adv_acc_answered_only'])

            if not subset.empty:
                fig, ax = plt.subplots(figsize=(8, 5))
                x = np.arange(len(subset.index))
                width = 0.38

                raw_vals = subset['adv_acc'].values * 100
                ans_vals = subset['adv_acc_answered_only'].values * 100

                bars1 = ax.bar(x - width/2, raw_vals, width, label='ADV raw', color='#2C3E50', alpha=0.9)
                bars2 = ax.bar(x + width/2, ans_vals, width, label='ADV answered-only', color='#E67E22', alpha=0.9)

                ax.set_ylabel('Accuracy (%)', weight='bold')
                ax.set_title('Figure R04 — Legal-CoT:\nRaw vs Answered-Only Accuracy', weight='bold')
                ax.set_xticks(x)
                ax.set_xticklabels(subset.index, rotation=15)
                ax.set_ylim(0, 115)
                ax.legend(frameon=False)

                for bars in [bars1, bars2]:
                    for b in bars:
                        ax.text(
                            b.get_x() + b.get_width()/2,
                            b.get_height() + 1,
                            f'{b.get_height():.1f}%',
                            ha='center',
                            fontsize=9,
                            fontweight='bold'
                        )

                plt.tight_layout()
                p = FIGURES_DIR / 'figR04_cot_raw_vs_answered.png'
                fig.savefig(p, bbox_inches='tight', facecolor='white')
                plt.show()
                print(f'Saved -> {p}')

## Section 6 — Qualitative Error Taxonomy\nExtracts instances where top LLMs failed in JACV-ADV for manual/LLM-assisted behavioral analysis.


In [ ]:
target_model = 'Claude-Sonnet-4'

adv_error_csv = TABLES_DIR / f'adv_errors_{target_model}.csv'
metrics_csv   = TABLES_DIR / 'publication_model_metrics.csv'
metrics_tex   = TABLES_DIR / 'publication_model_metrics.tex'

pub_df = df_metrics.copy()
pub_df.to_csv(metrics_csv, encoding='utf-8-sig')

latex_cols = [
    c for c in [
        'type',
        'cls_acc', 'cls_f1_macro', 'adv_acc',
        'adv_acc_answered_only', 'inconclusive',
        'cls_acc_ci95', 'cls_f1_ci95', 'adv_acc_ci95'
    ] if c in pub_df.columns
]

with open(metrics_tex, 'w', encoding='utf-8') as f:
    f.write(pub_df[latex_cols].to_latex(float_format='%.4f'))

print(f'Saved CSV -> {metrics_csv}')
print(f'Saved TEX -> {metrics_tex}')

target_ckpt_path = CKPT_DIR / f'ckpt_{target_model}.json'

if target_ckpt_path.exists():
    with open(target_ckpt_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    adv_preds = data.get('adv', {})

    errors = []
    for k, v in adv_preds.items():
        gold = v.get('gold')
        pred = v.get('pred')
        raw  = v.get('raw', '')

        if gold != pred:
            errors.append({
                'jacv_id': k,
                'gold': gold,
                'pred': pred,
                'raw_output': raw
            })

    df_errors = pd.DataFrame(errors)
    df_errors.to_csv(adv_error_csv, index=False, encoding='utf-8-sig')

    print(f'\nModel {target_model} made {len(df_errors)} errors in JACV-ADV.')
    print(f'Saved -> {adv_error_csv}')

    if not df_errors.empty:
        print('\n=== SAMPLE OF ADV ERRORS ===')
        print(df_errors.head(10).to_string(index=False))
else:
    print(f'Checkpoint for {target_model} not found: {target_ckpt_path}')